In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class AlcoholPhenolSulfation(MorphingOperator):
    def __init__(self):
        super(AlcoholPhenolSulfation, self).__init__()
        self._name = "O-Sulfation (Alcohols/Phenols)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[OX2H][#6;!$(C=O)]")

    def setOriginal(self, mol):
        super(AlcoholPhenolSulfation, self).setOriginal(mol)
        self._target_atoms = []

        if not self.original:
            return

        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None:
            return

        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original:
            return None

        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None:
            return None

        if not self._target_atoms:
            return MolpherMol(other=rdkit_mol)

        oxygen_idx = random.choice(self._target_atoms)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)

            # Εισαγωγή ουδέτερης ομάδας -SO3H
            sulfur_idx = rw_mol.AddAtom(Chem.Atom(16)) # S
            rw_mol.AddBond(oxygen_idx, sulfur_idx, Chem.BondType.SINGLE)

            o1_idx = rw_mol.AddAtom(Chem.Atom(8)) # =O
            rw_mol.AddBond(sulfur_idx, o1_idx, Chem.BondType.DOUBLE)

            o2_idx = rw_mol.AddAtom(Chem.Atom(8)) # =O
            rw_mol.AddBond(sulfur_idx, o2_idx, Chem.BondType.DOUBLE)

            o3_idx = rw_mol.AddAtom(Chem.Atom(8)) # -OH (Ουδέτερο)
            rw_mol.AddBond(sulfur_idx, o3_idx, Chem.BondType.SINGLE)

            new_mol = rw_mol.GetMol()

            for idx in [oxygen_idx, sulfur_idx, o1_idx, o2_idx, o3_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                if atom.GetAtomicNum() != 16: # Αφήνουμε το Θείο να διαχειριστεί το σθένος του (6)
                    atom.SetNoImplicit(False)
                    atom.SetNumExplicitHs(0)

            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)

            return MolpherMol(other=new_mol)

        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name

sulfation_op = AlcoholPhenolSulfation()

test_molecules_sulf = {
    "1. Παρακεταμόλη (Φαινόλη)": "CC(=O)Nc1ccc(O)cc1",
    "2. Βενζοϊκό οξύ (Μόνο Καρβοξύλιο)": "O=C(O)c1ccccc1",
    "3. 1-Βουτανόλη (Αλειφατική Αλκοόλη)": "CCCCO",
    "4. Σαλικυλικό Οξύ (Φαινόλη + Οξύ)": "O=C(O)c1ccccc1O"
}

print("=== STARTING SULFATION TESTING ===")
for name, smiles in test_molecules_sulf.items():
    test_mol = Chem.MolFromSmiles(smiles)
    if test_mol is None:
        print(f"\n{name}\n  SMILES Parse Error!")
        continue
        
    mol = MolpherMol(smiles)
    sulfation_op.setOriginal(mol)
    product = sulfation_op.morph()
    
    print(f"\n{name}")
    print(f"  SOURCE: {mol.getSMILES()}")
    
    # Αν το target SMILES είναι ολόιδιο με το source, σημαίνει ότι δεν έγινε αντίδραση
    if product and product.getSMILES() != mol.getSMILES():
        print(f"  TARGET: {product.getSMILES()}")
    else:
        print("  TARGET: No change (Safe - Ignored)")
print("\n==================================")

=== STARTING SULFATION TESTING ===

1. Παρακεταμόλη (Φαινόλη)
  SOURCE: CC(=O)NC1=CC=C(O)C=C1
  TARGET: CC(=O)NC1=CC=C(OS(=O)(=O)O)C=C1

2. Βενζοϊκό οξύ (Μόνο Καρβοξύλιο)
  SOURCE: O=C(O)C1=CC=CC=C1
  TARGET: No change (Safe - Ignored)

3. 1-Βουτανόλη (Αλειφατική Αλκοόλη)
  SOURCE: CCCCO
  TARGET: CCCCOS(=O)(=O)O

4. Σαλικυλικό Οξύ (Φαινόλη + Οξύ)
  SOURCE: O=C(O)C1=CC=CC=C1O
  TARGET: O=C(O)C1=CC=CC=C1OS(=O)(=O)O

